# 🧩 Mini-Lab: First Function Call

**Module 7: AI Agents** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** how function calling works in OpenAI's API
2. **Define** tools using JSON schema format that the LLM can invoke
3. **Identify** the structure of tool schemas including parameter types, descriptions, and required fields
4. **Implement** a complete function-calling round trip: request → tool call → result → final response

## Target Concepts

| Concept | Description |
|---------|-------------|
| Function Calling | OpenAI's mechanism for the LLM to request execution of external functions |
| Tool Definition | Defining tools with name, description, and JSON schema so the LLM knows what's available |
| Tool Schemas | Parameter specifications using JSON Schema (types, descriptions, enums, required fields) |

## Setup

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import json

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## What Is Function Calling?

LLMs can generate text, but they can't actually **do** things — they can't check the weather, query a database, or send an email. **Function calling** bridges this gap.

Here's how it works:

1. You tell the LLM what **tools** (functions) are available
2. The LLM decides **if and when** to call a tool based on the user's request
3. The LLM returns a **structured request** to call a specific function with specific arguments
4. **You** execute the function and send the result back
5. The LLM uses the result to generate a final response

> The LLM never executes code — it only **requests** that you execute it.

## Step 1: Define a Python Function

Let's start with a simple function that the LLM will be able to call — getting the current weather for a city.

In [3]:
def get_weather(city, unit="fahrenheit"):
    """Simulate getting weather data for a city."""
    # In a real app, this would call a weather API
    weather_data = {
        "new york": {"temp": 72, "condition": "sunny"},
        "london": {"temp": 58, "condition": "cloudy"},
        "tokyo": {"temp": 80, "condition": "humid"},
    }
    
    data = weather_data.get(city.lower(), {"temp": 65, "condition": "unknown"})
    
    if unit == "celsius":
        data["temp"] = round((data["temp"] - 32) * 5 / 9)
    
    return json.dumps({"city": city, "temperature": data["temp"], "unit": unit, "condition": data["condition"]})

# Test it directly
print(get_weather("London", "celsius"))

{"city": "London", "temperature": 14, "unit": "celsius", "condition": "cloudy"}


## Step 2: Define the Tool Schema

Now we need to tell the LLM about this function. We do this by creating a **tool definition** — a JSON schema that describes:

- **name**: The function name the LLM will reference
- **description**: What the function does (helps the LLM decide when to use it)
- **parameters**: A JSON Schema object describing each parameter — its type, description, allowed values, and whether it's required

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'New York', 'London'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["fahrenheit", "celsius"],
                        "description": "Temperature unit. Defaults to fahrenheit."
                    }
                },
                "required": ["city"]
            }
        }
    }
]

md("**Tool defined!** Let's inspect its structure:")
print(json.dumps(tools, indent=2))

**Tool defined!** Let's inspect its structure:

[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a given city.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The city name, e.g., 'New York', 'London'"
          },
          "unit": {
            "type": "string",
            "enum": [
              "fahrenheit",
              "celsius"
            ],
            "description": "Temperature unit. Defaults to fahrenheit."
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


### Anatomy of the Tool Schema

Let's break down what each part does:

| Field | Purpose |
|-------|--------|
| `type: "function"` | Tells OpenAI this is a function tool (vs. other tool types) |
| `name` | Identifier the LLM uses when requesting a call |
| `description` | Helps the LLM decide **when** this tool is relevant |
| `parameters.type` | Always `"object"` — parameters are passed as a JSON object |
| `properties` | Each parameter with its type and description |
| `enum` | Restricts a parameter to specific allowed values |
| `required` | Lists parameters that must be provided |

## Step 3: Send a Request with Tools

Now let's ask the LLM a question that should trigger it to call our function.

In [5]:
messages = [
    {"role": "user", "content": "What's the weather like in Tokyo?"}
]

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Tool calls:   {assistant_message.tool_calls}")

Finish reason: tool_calls
Tool calls:   [ChatCompletionMessageFunctionToolCall(id='call_XMIjhuUd3SlHg9nB7b1VUCbq', function=Function(arguments='{"city":"Tokyo"}', name='get_weather'), type='function')]


The `finish_reason` is `"tool_calls"` instead of the usual `"stop"` — this means the LLM wants us to execute a function before it can answer.

Let's inspect what the LLM is asking us to do:

In [6]:
tool_call = assistant_message.tool_calls[0]

md(f"""
### Tool Call Details

| Field | Value |
|-------|-------|
| **Tool Call ID** | `{tool_call.id}` |
| **Function Name** | `{tool_call.function.name}` |
| **Arguments** | `{tool_call.function.arguments}` |
""")


### Tool Call Details

| Field | Value |
|-------|-------|
| **Tool Call ID** | `call_XMIjhuUd3SlHg9nB7b1VUCbq` |
| **Function Name** | `get_weather` |
| **Arguments** | `{"city":"Tokyo"}` |


The LLM extracted `"Tokyo"` from the user's natural language message and structured it as a proper function argument. This is the core magic of function calling — **the LLM translates natural language into structured function calls**.

## Step 4: Execute the Function and Return Results

Now we complete the round trip:
1. Parse the arguments from the tool call
2. Execute our Python function
3. Send the result back to the LLM
4. Get the final natural-language response

In [7]:
# Parse the arguments the LLM provided
args = json.loads(tool_call.function.arguments)
print(f"Parsed arguments: {args}")

# Execute our function
result = get_weather(**args)
print(f"Function result:  {result}")

Parsed arguments: {'city': 'Tokyo'}
Function result:  {"city": "Tokyo", "temperature": 80, "unit": "fahrenheit", "condition": "humid"}


In [8]:
# Build the full conversation with the tool result
messages = [
    {"role": "user", "content": "What's the weather like in Tokyo?"},
    assistant_message,  # The LLM's response requesting the tool call
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    }
]

# Get the final response
final_response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools
)

md(f"**Assistant:** {final_response.choices[0].message.content}")

**Assistant:** The weather in Tokyo is humid with a temperature of around 80°F.

The LLM took the raw JSON from our function and turned it into a natural, human-friendly response.

## Step 5: The Complete Flow in One Place

Let's see the full function-calling pattern together — this is the pattern you'll reuse every time you work with tools.

In [9]:
# Map function names to actual Python functions
available_functions = {
    "get_weather": get_weather
}

def ask_with_tools(user_message):
    """Complete function-calling round trip."""
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: Send request with tools
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        tools=tools
    )
    
    assistant_msg = response.choices[0].message
    
    # Step 2: Check if the LLM wants to call a tool
    if assistant_msg.tool_calls:
        messages.append(assistant_msg)
        
        # Step 3: Execute each tool call
        for tc in assistant_msg.tool_calls:
            fn = available_functions[tc.function.name]
            args = json.loads(tc.function.arguments)
            result = fn(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        
        # Step 4: Get final response
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        return response.choices[0].message.content
    
    # No tool call needed — return direct response
    return assistant_msg.content

In [10]:
# Test with a weather question (should trigger tool call)
answer = ask_with_tools("How's the weather in London in celsius?")
md(f"**Weather question:** {answer}")

**Weather question:** The weather in London is currently 14°C and cloudy.

In [11]:
# Test with a non-weather question (should NOT trigger tool call)
answer = ask_with_tools("What is the capital of France?")
md(f"**Non-weather question:** {answer}")

**Non-weather question:** The capital of France is Paris.

Notice the LLM is smart enough to only call the tool when the question is relevant. For a general knowledge question, it simply answers directly.

## 🎯 Summary

### Key Takeaways

1. **Function calling** — lets the LLM request execution of your Python functions by generating structured arguments from natural language
2. **Tool definitions** — describe available functions using a `name`, `description`, and `parameters` schema so the LLM knows what it can call and when
3. **Tool schemas** — use JSON Schema to specify parameter types, descriptions, enums, and required fields for type-safe tool use
4. **The LLM never executes code** — it only produces a structured request; you execute the function and return the result
5. **Round trip pattern** — user message → LLM requests tool call → you execute → send result back → LLM generates final answer